# Notebook 7 — Encode Geospatial Network into the CANOE Schema

This notebook converts the geospatial outputs developed in previous notebooks into a CANOE-compatible SQLite database.

The objective is to replace the synthetic grid-neighbor representation used in the prototype model with a transport network derived from the Canadian basemap and road connectivity analysis while preserving compatibility with the existing CANOE/TEMOA model structure.

Transport links are represented using CANOE pseudo-regions of the form

```
region_from-region_to
```

where the two regions correspond to adjacent basemap polygons connected by existing infrastructure. This forms a dual graph representation of Canadian infrastructure layers aligned under a specified spatial resolution.

Initially, this notebook focuses on road-based transport technologies and encodes only links for which road connectivity has been identified. The absence of a link implies that transport between those regions is infeasible.

Rather than rebuilding the complete database from raw CSV files, this notebook loads the existing CANOE database and performs a schema reconciliation step. Inherited tables are filtered to the geospatial region topology and augmented with new transport technologies, producing a functional geospatial test database suitable for MILP execution.

The resulting database provides an intermediate development layer between the geospatial preprocessing workflow and eventual integration into the core CANOE modules.

---

## Inputs

### Basemap regions

From Notebook 4:

* Regional polygon geometries
* Region identifiers
* Region centroids

### Neighbor relationships

From Notebook 5:

* Polygon adjacency graph
* Neighbor pairs
* Inter-region distances

### Road connectivity

From Notebook 6:

* Weak road connectivity
* Strong road connectivity

### Existing CANOE database

* Existing CANOE SQLite database
* Schema definitions
* Technology definitions
* Commodity definitions
* Supporting tables

---

## Outputs

This notebook modifies and validates CANOE tables including:

* Region
* Technology
* Efficiency
* CostVariable
* CostInvest
* ETLSegment
* Demand
* LimitCapacity
* Supporting schema tables

and exports complete SQLite databases suitable for direct use by the CANOE/TEMOA solver.

---

## Conceptual workflow

1. Load the regional basemap and road connectivity outputs.
2. Load the existing CANOE database.
3. Replace the synthetic region representation with geospatial regions.
4. Build transport edges from connected neighboring regions.
5. Encode transport technologies for each valid edge.
6. Reconcile inherited node and edge tables with the geospatial topology.
7. Validate schema consistency.
8. Export SQLite databases.
9. Test MILP execution using the existing CANOE workflow.

This notebook serves as a graph-to-schema encoder and schema reconciliation layer between the geospatial preprocessing workflow and eventual integration into the core CANOE modules.

---

## Version note — v3 (2026-06-30)

v3 was created to resolve two defects identified during CO2_CAP debugging that v2 did not have isolated fixes for:

1. **CO2_CAP unit mismatch.** `LimitCapacity` (`tech_or_group = CO2_CAP`) was populated from `emissions_kt_co2e_per_year` without unit conversion, leaving CO2_CAP capacity/activity implicitly in kt. `CostVariable` for CO2_CAP was independently entered as `M$/t`. Because `CapacityToActivity` has no entry for CO2_CAP (TEMOA defaults `c2a = 1`), capacity and activity share the same numeric scale, so the kt/t mismatch propagated directly into the objective function as a ~1000x distortion in CO2_CAP's cost contribution. Fixed by converting kt → t at the point of ingestion (Cell 11, multiply by 1000) rather than at any downstream table, and by explicitly setting `units = "t"` in the `LimitCapacity` write (Cell 14) instead of leaving it `None`.

**Open items not resolved in v3, carried forward:**
- Whether `emissions_kt_co2e_per_year` is consumed independently downstream (NB8, `figure_generator.py`, cached intermediates) in a way that could now disagree with NB7's t-scale output — not yet checked.
- ELC_GEN `CostVariable` magnitudes (~$27,000/MWh implied) may indicate a separate, pipeline-wide dollar-scaling convention issue, unrelated to the CO2_CAP fix above — not yet audited.
- This is the second instance of a units inconsistency in this pipeline (the first being the NB8/v2 CO2_CAP capacity inflation hypothesis). There is no canonical units registry anywhere in the schema (`Commodity` table has no units field) — each table's `units` column is free text, unenforced by TEMOA's solver. This version note documents a symptom fix, not the structural gap.

- Core focus of work going forward will be standardizing units and making use of data columns in the SQL TEMOA schema to ensure new data additions and unit conversions throughout pre-processing remain consistent before SQL encoding.

---

In [ ]:
# =============================================================================
# Dependencies and project directories
# =============================================================================

from pathlib import Path
import shutil
import sqlite3

import numpy as np
import pandas as pd
import math
import geopandas as gpd
import matplotlib.pyplot as plt

import db_mgmt


PROJECT_ROOT = Path.cwd().parent

DATA_FILES = PROJECT_ROOT / "data_files"

RAW_BASEMAPS = DATA_FILES / "raw" / "basemaps"

PROCESSED_BASEMAPS = DATA_FILES / "processed" / "basemaps"
PROCESSED_GRAPH = DATA_FILES / "processed" / "graph"
PROCESSED_ROAD_CONNECTIVITY = DATA_FILES / "processed" / "road_connectivity"
PROCESSED_SCHEMA = DATA_FILES / "processed" / "schema"

PROCESSED_SCHEMA.mkdir(
    parents=True,
    exist_ok=True,
)

In [ ]:
# =============================================================================
# Select basemap, graph, and road connectivity inputs
# =============================================================================

BASEMAP_STEM = "canada_basemap_1deg_intersects"
CONNECTION_METHOD = "weak"

RAW_BASEMAP_PATH = RAW_BASEMAPS / "lpr_000b21a_e.shp"
RAW_SCHEMA_PATH = DATA_FILES / "canoe_dataset_schema.sql"
BASELINE_SQLITE_PATH = DATA_FILES / "CANOE_geospatial.sqlite"

BASEMAP_PATH = (
    PROCESSED_BASEMAPS
    / f"{BASEMAP_STEM}.gpkg"
)

GRAPH_NODE_PATH = (
    PROCESSED_GRAPH
    / f"{BASEMAP_STEM}_graph_nodes.gpkg"
)

GRAPH_EDGE_PATH = (
    PROCESSED_GRAPH
    / f"{BASEMAP_STEM}_graph_edges.csv"
)

ROAD_EDGE_CONNECTIONS_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / f"{BASEMAP_STEM}_road_connectivity_{CONNECTION_METHOD}_road_edge_connections.csv"
)

ROAD_EDGES_GPKG_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / f"{BASEMAP_STEM}_road_connectivity_{CONNECTION_METHOD}_road_edges.gpkg"
)

ROAD_REGION_OVERLAY_PATH = (
    PROCESSED_ROAD_CONNECTIVITY
    / f"{BASEMAP_STEM}_road_connectivity_road_region_overlay.gpkg"
)

OUTPUT_SQLITE_PATH = (
    PROCESSED_SCHEMA
    / f"CANOE_geospatial_{BASEMAP_STEM}_roads_{CONNECTION_METHOD}.sqlite"
)


input_paths = {
    "raw_basemap": RAW_BASEMAP_PATH,
    "processed_basemap": BASEMAP_PATH,
    "graph_nodes": GRAPH_NODE_PATH,
    "graph_edges": GRAPH_EDGE_PATH,
    "road_edge_connections": ROAD_EDGE_CONNECTIONS_PATH,
    "road_edges_gpkg": ROAD_EDGES_GPKG_PATH,
    "road_region_overlay": ROAD_REGION_OVERLAY_PATH,
    "raw_schema": RAW_SCHEMA_PATH,
    "baseline_sqlite": BASELINE_SQLITE_PATH,
}

missing_paths = {
    name: path
    for name, path in input_paths.items()
    if not path.exists()
}

if missing_paths:
    for name, path in missing_paths.items():
        print(f"Missing {name}: {path}")
    raise FileNotFoundError("One or more required input files are missing.")

print("Selected configuration:")
print(f"Basemap: {BASEMAP_STEM}")
print(f"Road method: {CONNECTION_METHOD}")

print("\nAll required input files found.")
print(f"Processed basemap: {BASEMAP_PATH.name}")
print(f"Graph nodes: {GRAPH_NODE_PATH.name}")
print(f"Graph edges: {GRAPH_EDGE_PATH.name}")
print(f"Road connections: {ROAD_EDGE_CONNECTIONS_PATH.name}")
print(f"Road edge geometry: {ROAD_EDGES_GPKG_PATH.name}")
print(f"Output SQLite: {OUTPUT_SQLITE_PATH.name}")

In [ ]:
# =============================================================================
# Load basemap, graph, road, and baseline database inputs
# =============================================================================

basemap = gpd.read_file(
    BASEMAP_PATH,
)

graph_nodes = gpd.read_file(
    GRAPH_NODE_PATH,
)

graph_edges = pd.read_csv(
    GRAPH_EDGE_PATH,
)

road_edge_connections = pd.read_csv(
    ROAD_EDGE_CONNECTIONS_PATH,
)

road_edges_gdf = gpd.read_file(
    ROAD_EDGES_GPKG_PATH,
)

db = db_mgmt.sqlite_to_dfs(
    BASELINE_SQLITE_PATH,
)

print(f"Basemap regions: {len(basemap):,} rows")
print(f"Graph nodes: {len(graph_nodes):,} rows")
print(f"Graph edges: {len(graph_edges):,} rows")
print(f"Road edge connections: {len(road_edge_connections):,} rows")
print(f"Road edge geometries: {len(road_edges_gdf):,} rows")
print(f"Baseline database tables: {len(db):,}")

print("\nCRS:")
print(f"Basemap: {basemap.crs}")
print(f"Graph nodes: {graph_nodes.crs}")
print(f"Road edge geometries: {road_edges_gdf.crs}")

In [ ]:
# -----------------------------------------------------------------------------
# Raw input tables from original encoder
# -----------------------------------------------------------------------------

SITES_PATH = DATA_FILES / "sites_full.csv"
DEMAND_PATH = DATA_FILES / "demand.csv"

CO2_CLEAN_DIR = DATA_FILES / "processed" / "emissions" / "co2_large_facilities_2024"
CO2_CSV_PATH = CO2_CLEAN_DIR / "co2_large_facilities_2024_clean.csv"
CO2_GPKG_PATH = CO2_CLEAN_DIR / "co2_large_facilities_2024_clean.gpkg"

TRANSPORT_TECHS_PATH = DATA_FILES / "transport_techs.csv"
GEN_EFFICIENCIES_PATH = DATA_FILES / "generation_efficiency.csv"
TECHNOLOGIES_PATH = DATA_FILES / "techs.csv"
COMMODITIES_PATH = DATA_FILES / "commodities.csv"

sites_raw = pd.read_csv(SITES_PATH)
demand_raw = pd.read_csv(DEMAND_PATH)

# Prefer GPKG because it preserves geometry and facility metadata cleanly.
co2_raw = gpd.read_file(
    CO2_GPKG_PATH,
    layer="co2_large_facilities_2024",
)

transport_techs_raw = pd.read_csv(TRANSPORT_TECHS_PATH)
gen_efficiencies_raw = pd.read_csv(GEN_EFFICIENCIES_PATH)
technologies_raw = pd.read_csv(TECHNOLOGIES_PATH)
commodities_raw = pd.read_csv(COMMODITIES_PATH)

print("Raw input tables loaded:")
print(f"sites_full.csv: {len(sites_raw):,}")
print(f"demand.csv: {len(demand_raw):,}")
print(f"co2_large_facilities_2024_clean.gpkg: {len(co2_raw):,}")
print(f"transport_techs.csv: {len(transport_techs_raw):,}")
print(f"generation_efficiency.csv: {len(gen_efficiencies_raw):,}")
print(f"techs.csv: {len(technologies_raw):,}")
print(f"commodities.csv: {len(commodities_raw):,}")

print("\nsites_full.csv columns:", sites_raw.columns.tolist())
print("demand.csv columns:    ", demand_raw.columns.tolist())
print("CO2 columns:           ", co2_raw.columns.tolist())

In [ ]:
# =============================================================================
# Inspect core database and graph inputs
# =============================================================================

core_tables = [
    "Region",
    "Technology",
    "TechnologyType",
    "Commodity",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
    "LimitCapacity",
    "Demand",
    "DataSet",
]

for table in core_tables:
    df = db[table]
    print(f"{table}: {len(df):,} rows")

print("\nGraph input columns:")
print(f"graph_nodes: {list(graph_nodes.columns)}")
print(f"graph_edges: {list(graph_edges.columns)}")
print(f"road_edge_connections: {list(road_edge_connections.columns)}")

display(graph_nodes.head())
display(graph_edges.head())
display(road_edge_connections.head())
display(db["Technology"].sort_values("tech").reset_index(drop=True))
display(db["Commodity"].sort_values("name").reset_index(drop=True))

In [ ]:
# =============================================================================
# Build canonical region and transport-link tables
# =============================================================================

region_table = (
    graph_nodes[["region"]]
    .drop_duplicates()
    .sort_values(
        "region",
        key=lambda s: s.str.extract(r"R(\d+)")[0].astype(int),
    )
    .reset_index(drop=True)
)

region_table["notes"] = (
    f"{BASEMAP_STEM} CANOE geospatial graph node"
)

road_links = (
    road_edge_connections
    .loc[road_edge_connections["has_road_connection"]]
    .copy()
)

road_links = road_links[
    [
        "edge_region",
        "region_from",
        "region_to",
        "direction",
        "connection_method",
        "distance_km",
        "lon_from",
        "lat_from",
        "lon_to",
        "lat_to",
    ]
].copy()

road_links = (
    road_links
    .drop_duplicates(subset=["edge_region"])
    .reset_index(drop=True)
)

road_links["canoe_region"] = road_links["edge_region"]

pipeline_links = graph_edges.copy()
pipeline_links["canoe_region"] = pipeline_links["edge_region"]

VALID_NODE_REGIONS = set(region_table["region"])
VALID_PIPELINE_EDGE_REGIONS = set(pipeline_links["canoe_region"])
VALID_ROAD_EDGE_REGIONS = set(road_links["canoe_region"])

print(f"Region table rows: {len(region_table):,}")
print(f"Pipeline links: {len(pipeline_links):,}")
print(f"Road links ({CONNECTION_METHOD}): {len(road_links):,}")

display(region_table.head())
display(pipeline_links.head())
display(road_links.head())

In [ ]:
# =============================================================================
# Validate canonical region and transport-link tables
# =============================================================================

assert "R-999" not in set(region_table["region"])
assert len(region_table) == region_table["region"].nunique()
assert len(region_table) == len(graph_nodes)

assert pipeline_links["canoe_region"].nunique() == len(pipeline_links)
assert road_links["canoe_region"].nunique() == len(road_links)

assert set(pipeline_links["region_from"]).issubset(VALID_NODE_REGIONS)
assert set(pipeline_links["region_to"]).issubset(VALID_NODE_REGIONS)

assert set(road_links["region_from"]).issubset(VALID_NODE_REGIONS)
assert set(road_links["region_to"]).issubset(VALID_NODE_REGIONS)

assert pipeline_links["distance_km"].notna().all()
assert road_links["distance_km"].notna().all()

assert (pipeline_links["distance_km"] > 0).all()
assert (road_links["distance_km"] > 0).all()

assert pipeline_links["canoe_region"].str.contains("-", regex=False).all()
assert road_links["canoe_region"].str.contains("-", regex=False).all()

print("Canonical region and transport-link tables validated.")

print("\nDistance summaries:")
display(
    pd.DataFrame(
        {
            "pipeline_km": pipeline_links["distance_km"].describe(),
            "road_km": road_links["distance_km"].describe(),
        }
    )
)

In [ ]:
# =============================================================================
# Initialize encoded database and define transport technologies from input tables
# =============================================================================

db_encoded = {
    table_name: df.copy()
    for table_name, df in db.items()
}

db_encoded["Region"] = region_table.copy()

transport_techs = transport_techs_raw.copy()

required_transport_cols = {
    "tech",
    "input_comm",
    "output_comm",
    "cost_per_km",
    "intercept_cost_per_km",
}

missing_transport_cols = required_transport_cols - set(transport_techs.columns)

if missing_transport_cols:
    raise ValueError(
        "transport_techs.csv is missing required columns: "
        f"{sorted(missing_transport_cols)}"
    )

pipeline_tech_specs = transport_techs.loc[
    transport_techs["tech"].str.endswith("_PIPE")
].copy()

truck_tech_specs = transport_techs.loc[
    transport_techs["tech"].str.endswith("_TRUCK")
].copy()

transmission_tech_specs = transport_techs.loc[
    transport_techs["tech"] == "ELC_TRANS"
].copy()

PIPE_TECHS = set(pipeline_tech_specs["tech"])
TRUCK_TECHS = set(truck_tech_specs["tech"])
TRANS_TECHS = set(transmission_tech_specs["tech"])
TRANSPORT_TECHS = PIPE_TECHS | TRUCK_TECHS | TRANS_TECHS

assert not pipeline_tech_specs.empty, "No *_PIPE rows found in transport_techs.csv."
assert not truck_tech_specs.empty, "No *_TRUCK rows found in transport_techs.csv."
assert not transmission_tech_specs.empty, "No ELC_TRANS row found in transport_techs.csv."

print(f"Baseline Region rows: {len(db['Region']):,}")
print(f"Encoded Region rows: {len(db_encoded['Region']):,}")
print(f"Pipeline techs: {sorted(PIPE_TECHS)}")
print(f"Truck techs: {sorted(TRUCK_TECHS)}")
print(f"Transmission techs: {sorted(TRANS_TECHS)}")

display(truck_tech_specs)
display(pipeline_tech_specs)
display(transmission_tech_specs)

In [ ]:
def snap_points_to_graph_nodes(
    points: pd.DataFrame | gpd.GeoDataFrame,
    graph_nodes: gpd.GeoDataFrame,
    lon_col: str = "lon",
    lat_col: str = "lat",
) -> pd.DataFrame:

    points_df = (
        pd.DataFrame(points.drop(columns="geometry"))
        if isinstance(points, gpd.GeoDataFrame)
        else points.copy()
    )

    points_gdf = gpd.GeoDataFrame(
        points_df,
        geometry=gpd.points_from_xy(points_df[lon_col], points_df[lat_col]),
        crs="EPSG:4326",
    )

    nodes_gdf = graph_nodes[["region", "geometry"]].copy()

    # Pass 1: point-in-polygon for interior points.
    snapped = gpd.sjoin(
        points_gdf,
        nodes_gdf,
        how="left",
        predicate="within",
    ).drop(columns="index_right")

    # Pass 2: nearest-neighbour fallback for unmatched boundary/coastal points.
    unmatched_mask = snapped["region"].isna()
    n_unmatched = unmatched_mask.sum()

    if n_unmatched > 0:
        print(f"Pass 1: {n_unmatched:,} points outside all polygons — applying nearest-neighbour fallback.")

        unmatched_gdf = points_gdf.loc[unmatched_mask].copy()

        snapped_nearest = gpd.sjoin_nearest(
            unmatched_gdf.to_crs("EPSG:3347"),
            nodes_gdf.to_crs("EPSG:3347"),
            how="left",
            distance_col="snap_distance_m",
        ).to_crs("EPSG:4326")

        # sjoin_nearest can produce duplicate rows when a point is equidistant
        # from multiple polygons. Keep the first match per input index.
        snapped_nearest = (
            snapped_nearest
            .loc[~snapped_nearest.index.duplicated(keep="first")]
        )

        snapped.loc[unmatched_mask, "region"] = snapped_nearest["region"].values

        print(f"Pass 2: fallback assigned {n_unmatched:,} points.")
        print(f"        Max snap distance: {snapped_nearest['snap_distance_m'].max():,.0f} m")
        print(f"        Mean snap distance: {snapped_nearest['snap_distance_m'].mean():,.0f} m")

    return pd.DataFrame(snapped.drop(columns="geometry"))

In [ ]:
# -----------------------------------------------------------------------------
# Snap original electricity and fuel demand point inputs
# -----------------------------------------------------------------------------

raw_points_non_co2 = pd.concat(
    [
        sites_raw,
        demand_raw,
    ],
    ignore_index=True,
)

numeric_cols = raw_points_non_co2.select_dtypes(include="number").columns
raw_points_non_co2[numeric_cols] = raw_points_non_co2[numeric_cols].fillna(0)

snapped_non_co2 = snap_points_to_graph_nodes(
    raw_points_non_co2,
    graph_nodes,
)

In [ ]:
# -----------------------------------------------------------------------------
# Snap cleaned 2024 CO2 facility GPKG input
# -----------------------------------------------------------------------------

co2_facilities = co2_raw.copy()

co2_facilities["co2"] = pd.to_numeric(
    co2_facilities["emissions_kt_co2e_per_year"],
    errors="coerce",
).fillna(0) * 1000  # kt -> t

print(
    f"Units converted: emissions_kt_co2e_per_year (kt) -> co2 (t), "
    f"factor = 1000"
)

co2_facilities = co2_facilities.loc[
    co2_facilities["co2"] > 0
].copy()

snapped_co2 = snap_points_to_graph_nodes(
    co2_facilities,
    graph_nodes,
    lon_col="longitude",
    lat_col="latitude",
)

co2_region = (
    snapped_co2
    .groupby("region", as_index=False)
    .agg(
        co2=("co2", "sum"),
        n_co2_facilities=("facility_id", "count"),
    )
)

unmatched_co2_mask = snapped_co2["region"].isna()

# snapped_co2 region should be fully assigned after pass 2 — so check snap distance indirectly
# by re-running nearest on the original unmatched set
co2_facilities_reset = co2_facilities.reset_index(drop=True)

unmatched_co2 = co2_facilities_reset.loc[
    snapped_co2.reset_index(drop=True)["region"].isna()
].copy()

print(f"Still unmatched after fallback: {len(unmatched_co2):,}")

# Reconstruct which facilities were in the fallback path
# by checking snap distance via a fresh sjoin_nearest on all co2_facilities
co2_gdf = gpd.GeoDataFrame(
    co2_facilities_reset,
    geometry=gpd.points_from_xy(co2_facilities_reset["longitude"], co2_facilities_reset["latitude"]),
    crs="EPSG:4326",
)

nodes_gdf = graph_nodes[["region", "geometry"]].copy()

co2_nearest = gpd.sjoin_nearest(
    co2_gdf.to_crs("EPSG:3347"),
    nodes_gdf.to_crs("EPSG:3347"),
    how="left",
    distance_col="snap_distance_m",
).to_crs("EPSG:4326")

co2_nearest = co2_nearest.loc[~co2_nearest.index.duplicated(keep="first")]

far_co2 = co2_nearest.loc[co2_nearest["snap_distance_m"] > 50000].copy()

print(f"CO2 facilities snapped > 50 km: {len(far_co2):,}")
print(f"Total emissions in far facilities: {far_co2['emissions_kt_co2e_per_year'].sum():,.2f} kt CO2e/year")
print(f"Total emissions all facilities: {co2_facilities['emissions_kt_co2e_per_year'].sum():,.2f} kt CO2e/year")
print(f"Far fraction: {far_co2['emissions_kt_co2e_per_year'].sum() / co2_facilities['emissions_kt_co2e_per_year'].sum():.2%}")
display(
    far_co2[["facility_name", "province", "longitude", "latitude", "snap_distance_m", "emissions_kt_co2e_per_year", "region"]]
    .sort_values("snap_distance_m", ascending=False)
    .reset_index(drop=True)
    .head(20)
)

In [ ]:
# -----------------------------------------------------------------------------
# Aggregate snapped inputs to graph-node regions
# -----------------------------------------------------------------------------

site_attributes_non_co2 = (
    snapped_non_co2
    .groupby("region", as_index=False)
    .agg(
        LCOE=("LCOE", "mean"),
        max_elc=("max_elec", "sum"),
        demand=("demand", "sum"),
    )
)

site_attributes = (
    region_table[["region"]]
    .merge(site_attributes_non_co2, on="region", how="left")
    .merge(co2_region, on="region", how="left")
    .fillna(
        {
            "LCOE": 0,
            "max_elc": 0,
            "demand": 0,
            "co2": 0,
            "n_co2_facilities": 0,
        }
    )
)

site_attributes["co2_cost"] = 50

print(f"Snapped site attribute rows: {len(site_attributes):,}")
print(f"Regions with demand: {(site_attributes['demand'] > 0).sum():,}")
print(f"Regions with CO2: {(site_attributes['co2'] > 0).sum():,}")
print(f"CO2 facilities mapped: {int(site_attributes['n_co2_facilities'].sum()):,}")
print(f"Total CO2 mapped: {site_attributes['co2'].sum():,.2f} t CO2e/year")
print(f"Regions with electricity potential: {(site_attributes['max_elc'] > 0).sum():,}")
print(f"Facilities dropped (zero/negative emissions): {len(co2_raw) - len(co2_facilities):,}")

display(site_attributes.head())

In [ ]:
# =============================================================================
# Replace node-level Demand and LimitCapacity from snapped attributes
# =============================================================================

demand_sites = (
    site_attributes
    .loc[site_attributes["demand"] > 0]
    .reset_index(drop=True)
)

db_encoded["Demand"] = pd.DataFrame(
    {
        "region": demand_sites["region"],
        "period": 1,
        "commodity": "d_gsl",
        "demand": demand_sites["demand"],
        "units": None,
        "notes": "Demand snapped to selected geospatial graph node",
        "data_source": None,
        "dq_cred": None,
        "dq_geog": None,
        "dq_struc": None,
        "dq_tech": None,
        "dq_time": None,
        "data_id": "GEO001",
    }
)

db_encoded["LimitCapacity"] = pd.concat(
    [
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech_or_group": "CO2_CAP",
                "operator": "le",
                "capacity": site_attributes["co2"],
                "units": "t",
                "notes": "CO2 capacity snapped to selected geospatial graph node",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech_or_group": "ELC_GEN",
                "operator": "le",
                "capacity": site_attributes["max_elc"],
                "units": None,
                "notes": "Electricity potential snapped to selected geospatial graph node",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
    ],
    ignore_index=True,
)

assert len(db_encoded["Demand"]) == len(demand_sites)
assert db_encoded["Demand"]["region"].nunique() == len(db_encoded["Demand"])
assert db_encoded["Demand"]["demand"].gt(0).all()
assert len(db_encoded["LimitCapacity"]) == 2 * len(site_attributes)

print(f"Demand rows: {len(db_encoded['Demand']):,}")
print(f"LimitCapacity rows: {len(db_encoded['LimitCapacity']):,}")

In [ ]:
# =============================================================================
# Rebuild node-level CostVariable and CostInvest rows
# =============================================================================

NODE_COSTVARIABLE_TECHS = [
    "ELC_GEN",
    "CO2_CAP",
    "GSL_BACKUP",
]

node_costvariable = pd.concat(
    [
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech": "ELC_GEN",
                "vintage": 1,
                "cost": site_attributes["LCOE"],
                "units": "M$/MWh",
                "notes": "Electricity generation cost snapped to selected graph node",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech": "CO2_CAP",
                "vintage": 1,
                "cost": site_attributes["co2_cost"],
                "units": "M$/t",
                "notes": "CO2 capture cost snapped to selected graph node",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "period": 1,
                "tech": "GSL_BACKUP",
                "vintage": 1,
                "cost": 500000,
                "units": "M$/MWh",
                "notes": "Backup gasoline supply cost",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
    ],
    ignore_index=True,
)

db_encoded["CostVariable"] = (
    db_encoded["CostVariable"]
    .loc[
        ~db_encoded["CostVariable"]["tech"].isin(NODE_COSTVARIABLE_TECHS)
    ]
    .copy()
)

db_encoded["CostVariable"] = pd.concat(
    [
        db_encoded["CostVariable"],
        node_costvariable,
    ],
    ignore_index=True,
)


NODE_COSTINVEST_TECHS = [
    "ELC_GEN",
    "CO2_CAP",
]

node_costinvest = pd.concat(
    [
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "tech": "ELC_GEN",
                "vintage": 1,
                "cost": 1000,
                "units": None,
                "notes": "Electricity generation fixed investment cost",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
        pd.DataFrame(
            {
                "region": site_attributes["region"],
                "tech": "CO2_CAP",
                "vintage": 1,
                "cost": 1000,
                "units": None,
                "notes": "CO2 capture fixed investment cost",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        ),
    ],
    ignore_index=True,
)

db_encoded["CostInvest"] = (
    db_encoded["CostInvest"]
    .loc[
        ~db_encoded["CostInvest"]["tech"].isin(NODE_COSTINVEST_TECHS)
    ]
    .copy()
)

db_encoded["CostInvest"] = pd.concat(
    [
        db_encoded["CostInvest"],
        node_costinvest,
    ],
    ignore_index=True,
)

print(f"Node CostVariable rows: {len(node_costvariable):,}")
print(f"Node CostInvest rows: {len(node_costinvest):,}")

In [ ]:
# =============================================================================
# Rebuild LimitTechInputSplitAnnual rows
# =============================================================================

def build_input_split(
    regions: pd.Series,
    tech: str,
    input_comm: list[str],
    proportion: list[float],
    operator: str = "ge",
) -> pd.DataFrame:

    if len(input_comm) != len(proportion):
        raise ValueError("input_comm and proportion must have same length.")

    if not math.isclose(sum(proportion), 1.0, rel_tol=1e-6):
        raise ValueError(
            f"Proportions for {tech} sum to {sum(proportion):.10f}, expected 1.0."
        )

    rows = []

    for comm, prop in zip(input_comm, proportion):

        rows.append(
            pd.DataFrame(
                {
                    "region": regions,
                    "period": 1,
                    "input_comm": comm,
                    "tech": tech,
                    "operator": operator,
                    "proportion": prop,
                    "notes": None,
                    "data_source": None,
                    "dq_cred": None,
                    "dq_geog": None,
                    "dq_struc": None,
                    "dq_tech": None,
                    "dq_time": None,
                    "data_id": "GEO001",
                }
            )
        )

    return pd.concat(rows, ignore_index=True)


node_regions = site_attributes["region"]

gsl_input_split = build_input_split(
    node_regions,
    "GSL_PLANT",
    ["ch3oh", "h2"],
    [0.997782705, 0.002217295],
)

metoh_input_split = build_input_split(
    node_regions,
    "METOH_PLANT",
    ["co2", "h2", "elc"],
    [0.79230333899, 0.10865874363, 0.09903791737],
)

db_encoded["LimitTechInputSplitAnnual"] = pd.concat(
    [
        gsl_input_split,
        metoh_input_split,
    ],
    ignore_index=True,
)

print(
    f"LimitTechInputSplitAnnual rows: "
    f"{len(db_encoded['LimitTechInputSplitAnnual']):,}"
)

In [ ]:
# =============================================================================
# Rebuild node-level production and demand Efficiency rows
# =============================================================================

efficiency_rows = []

for row in gen_efficiencies_raw.itertuples(index=False):

    if row.tech == "GSL_BACKUP":
        regions = (
            site_attributes
            .loc[site_attributes["demand"] > 0, "region"]
            .reset_index(drop=True)
        )
    else:
        regions = site_attributes["region"].reset_index(drop=True)

    efficiency_rows.append(
        pd.DataFrame(
            {
                "region": regions,
                "input_comm": row.input_comm,
                "tech": row.tech,
                "vintage": 1,
                "output_comm": row.output_comm,
                "efficiency": row.efficiency,
                "notes": "Node-level efficiency rebuilt from snapped graph regions",
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        )
    )

node_efficiency = pd.concat(
    efficiency_rows,
    ignore_index=True,
)

demand_regions = (
    db_encoded["Demand"]["region"]
    .drop_duplicates()
    .reset_index(drop=True)
)

gsl_demand_efficiency = pd.DataFrame(
    {
        "region": demand_regions,
        "input_comm": "gsl",
        "tech": "GSL_DEMAND",
        "vintage": 1,
        "output_comm": "d_gsl",
        "efficiency": 1.0,
        "notes": "Gasoline demand technology rebuilt from snapped demand regions",
        "data_source": None,
        "dq_cred": None,
        "dq_geog": None,
        "dq_struc": None,
        "dq_tech": None,
        "dq_time": None,
        "data_id": "GEO001",
    }
)

node_efficiency = pd.concat(
    [
        node_efficiency,
        gsl_demand_efficiency,
    ],
    ignore_index=True,
)

NODE_EFFICIENCY_TECHS = set(node_efficiency["tech"])

# Remove inherited rows for node-level technologies and all inherited edge rows.
# Edge rows are rebuilt later from the selected graph/road connectivity.
db_encoded["Efficiency"] = (
    db_encoded["Efficiency"]
    .loc[
        (
            ~db_encoded["Efficiency"]["tech"].isin(NODE_EFFICIENCY_TECHS)
        )
        &
        (
            ~db_encoded["Efficiency"]["region"]
            .astype(str)
            .str.contains("-", regex=False)
        )
    ]
    .copy()
)

db_encoded["Efficiency"] = pd.concat(
    [
        db_encoded["Efficiency"],
        node_efficiency,
    ],
    ignore_index=True,
)

assert set(db_encoded["Demand"]["region"]) == set(
    db_encoded["Efficiency"]
    .loc[
        db_encoded["Efficiency"]["tech"] == "GSL_DEMAND",
        "region",
    ]
)

assert not (
    db_encoded["Efficiency"]["region"]
    .astype(str)
    .str.contains("-", regex=False)
).any(), (
    "Edge-region Efficiency rows should not exist yet. "
    "Transport Efficiency rows must be rebuilt later from selected graph links."
)

print(f"Node Efficiency rows: {len(node_efficiency):,}")
print(f"Encoded Efficiency rows after node rebuild: {len(db_encoded['Efficiency']):,}")

In [ ]:
# =============================================================================
# Rebuild ETLSegment rows for plants and pipelines
# =============================================================================

PLANT_TECHS = {
    "GSL_PLANT",
    "METOH_PLANT",
}


# -----------------------------------------------------------------------------
# Build plant ETLSegment rows for selected node regions
# -----------------------------------------------------------------------------

plant_etl_template = (
    db["ETLSegment"]
    .loc[
        db["ETLSegment"]["tech_or_group"].isin(PLANT_TECHS)
    ]
    .copy()
)

assert not plant_etl_template.empty, (
    "No plant ETLSegment rows found in baseline database."
)

plant_etl_rows = []

for tech in sorted(PLANT_TECHS):

    tech_segments = (
        plant_etl_template
        .loc[plant_etl_template["tech_or_group"] == tech]
        .drop(columns=["region"])
        .drop_duplicates()
        .sort_values(["tech_or_group", "segment"])
        .reset_index(drop=True)
    )

    assert not tech_segments.empty, (
        f"No ETLSegment template rows found for {tech}."
    )

    region_frame = pd.DataFrame(
        {
            "region": site_attributes["region"],
            "key": 1,
        }
    )

    segment_frame = tech_segments.copy()
    segment_frame["key"] = 1

    df = (
        region_frame
        .merge(segment_frame, on="key")
        .drop(columns="key")
    )

    plant_etl_rows.append(df)

plant_etl_new = pd.concat(
    plant_etl_rows,
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# Build pipeline and transmission ETLSegment rows for selected graph edge regions
# Preserve each technology's own baseline segment curve, but scale
# investment cost by edge distance. The baseline ETLSegment curve from
# invest_costs() encodes economies of scale in volume only — it has no
# distance dependence, since it was built against a synthetic grid where
# adjacency distance was not the driving cost factor. On the real geospatial
# basemap, edge distances range ~14-111 km, so applying the baseline curve
# verbatim makes every edge cost the same to build regardless of
# length. Scale cost_lower/cost_upper by distance_km relative to the
# baseline's reference distance to restore a distance-sensitive investment
# cost while preserving the volume-based segment shape.
#
# ELC_TRANS is included here alongside the *_PIPE techs since it is built
# from its own invest_costs() call in model_run.py (a=2000, b=-0.3,
# transp_tech=True) and rides the same graph-edge adjacency as the pipes.
# -----------------------------------------------------------------------------

REFERENCE_DISTANCE_KM = pipeline_links["distance_km"].mean()

print(f"Reference distance for ETLSegment cost scaling: {REFERENCE_DISTANCE_KM:.2f} km")

pipeline_etl_rows = []

edge_tech_specs = pd.concat(
    [pipeline_tech_specs, transmission_tech_specs],
    ignore_index=True,
)

for pipe in edge_tech_specs.itertuples(index=False):

    pipe_etl_template = (
        db["ETLSegment"]
        .loc[
            db["ETLSegment"]["tech_or_group"] == pipe.tech
        ]
        .copy()
    )

    assert not pipe_etl_template.empty, (
        f"No ETLSegment rows found in baseline database for {pipe.tech}."
    )

    assert pipe_etl_template["segment"].nunique() > 1, (
        f"{pipe.tech} ETLSegment template has only one segment. "
        "This would collapse the piecewise investment formulation."
    )

    pipe_segments = (
        pipe_etl_template
        .drop(columns=["region"])
        .drop_duplicates()
        .sort_values(["tech_or_group", "segment"])
        .reset_index(drop=True)
    )

    edge_frame = pipeline_links[["canoe_region", "distance_km"]].rename(
        columns={"canoe_region": "region"}
    ).copy()
    edge_frame["key"] = 1

    segment_frame = pipe_segments.copy()
    segment_frame["key"] = 1

    df = (
        edge_frame
        .merge(segment_frame, on="key")
        .drop(columns="key")
    )

    # Scale investment cost by this edge's distance relative to the
    # reference distance. The volume-based segment shape (cap_lower,
    # cap_upper boundaries) is unchanged; only the cost magnitude scales.
    distance_factor = df["distance_km"] / REFERENCE_DISTANCE_KM

    df["cost_lower"] = df["cost_lower"] * distance_factor
    df["cost_upper"] = df["cost_upper"] * distance_factor

    df = df[
        [
            "region",
            "tech_or_group",
            "segment",
            "cap_lower",
            "cap_upper",
            "cost_lower",
            "cost_upper",
            "data_id",
        ]
    ].copy()

    pipeline_etl_rows.append(df)

pipeline_etl_new = pd.concat(
    pipeline_etl_rows,
    ignore_index=True,
)

# Validate the distance scaling actually produced variation in cost.
etl_cost_range = (
    pipeline_etl_new
    .groupby("tech_or_group")["cost_upper"]
    .agg(["min", "max"])
)

print("\nPipeline/transmission ETLSegment cost_upper range after distance scaling:")
display(etl_cost_range)

assert (etl_cost_range["max"] > etl_cost_range["min"]).all(), (
    "Distance scaling did not produce cost variation across edges — "
    "check that distance_km has meaningful spread."
)


# -----------------------------------------------------------------------------
# Replace ETLSegment geography
# -----------------------------------------------------------------------------

non_edge_etl = (
    db_encoded["ETLSegment"]
    .loc[
        ~db_encoded["ETLSegment"]["region"]
        .astype(str)
        .str.contains("-", regex=False)
    ]
    .copy()
)

non_rebuilt_node_etl = (
    non_edge_etl
    .loc[
        ~non_edge_etl["tech_or_group"].isin(PLANT_TECHS)
    ]
    .copy()
)

db_encoded["ETLSegment"] = pd.concat(
    [
        non_rebuilt_node_etl,
        plant_etl_new,
        pipeline_etl_new,
    ],
    ignore_index=True,
)


# -----------------------------------------------------------------------------
# Validate ETLSegment rebuild
# -----------------------------------------------------------------------------

assert (
    db_encoded["ETLSegment"][
        [
            "region",
            "tech_or_group",
            "segment",
        ]
    ]
    .duplicated()
    .sum()
    == 0
), "Duplicate ETLSegment primary keys found."

etl_edge_regions = set(
    db_encoded["ETLSegment"]
    .loc[
        db_encoded["ETLSegment"]["region"]
        .astype(str)
        .str.contains("-", regex=False),
        "region",
    ]
)

invalid_etl_edge_regions = sorted(
    etl_edge_regions
    - VALID_PIPELINE_EDGE_REGIONS
)

assert not invalid_etl_edge_regions, (
    "ETLSegment contains invalid edge regions: "
    f"{invalid_etl_edge_regions[:10]}"
)

print(f"Plant ETLSegment rows: {len(plant_etl_new):,}")
print(f"Pipeline/transmission ETLSegment rows: {len(pipeline_etl_new):,}")
print(f"Encoded ETLSegment rows: {len(db_encoded['ETLSegment']):,}")

display(
    db_encoded["ETLSegment"]
    .groupby("tech_or_group")["segment"]
    .nunique()
    .sort_index()
)

In [ ]:
# =============================================================================
# Rebuild Technology table from updated techs.csv
# =============================================================================

technology = technologies_raw.copy()

technology["sector"] = "industrial"
technology["reserve"] = 0
technology["curtail"] = 0
technology["retire"] = 0
technology["flex"] = 0
technology["data_id"] = "GEO001"

db_encoded["Technology"] = technology.copy()

assert TRUCK_TECHS.issubset(set(db_encoded["Technology"]["tech"])), (
    "Truck technologies are missing from techs.csv."
)

assert PIPE_TECHS.issubset(set(db_encoded["Technology"]["tech"])), (
    "Pipeline technologies are missing from techs.csv."
)

print(f"Technology rows: {len(db_encoded['Technology']):,}")
display(db_encoded["Technology"].sort_values("tech").reset_index(drop=True))

In [ ]:
# =============================================================================
# Build transport Efficiency rows
# =============================================================================

def build_transport_efficiency(
    links: pd.DataFrame,
    tech_specs: pd.DataFrame,
    notes: str,
) -> pd.DataFrame:

    rows = []

    for tech in tech_specs.itertuples(index=False):

        df = pd.DataFrame(
            {
                "region": links["canoe_region"],
                "input_comm": tech.input_comm,
                "tech": tech.tech,
                "vintage": 1,
                "output_comm": tech.output_comm,
                "efficiency": 1.0,
                "notes": notes,
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        )

        rows.append(df)

    return pd.concat(rows, ignore_index=True)


pipeline_efficiency = build_transport_efficiency(
    pipeline_links,
    pipeline_tech_specs,
    "Candidate pipeline transport link on canonical graph edge",
)

truck_efficiency = build_transport_efficiency(
    road_links,
    truck_tech_specs,
    f"Existing {CONNECTION_METHOD} road-connected transport link",
)

transmission_efficiency = build_transport_efficiency(
    pipeline_links,
    transmission_tech_specs,
    "Candidate electricity transmission link on canonical graph edge",
)

db_encoded["Efficiency"] = (
    db_encoded["Efficiency"]
    .loc[
        ~db_encoded["Efficiency"]["tech"].isin(TRANSPORT_TECHS)
    ]
    .copy()
)

db_encoded["Efficiency"] = pd.concat(
    [
        db_encoded["Efficiency"],
        pipeline_efficiency,
        truck_efficiency,
        transmission_efficiency,
    ],
    ignore_index=True,
)

assert set(pipeline_efficiency["region"]) == VALID_PIPELINE_EDGE_REGIONS
assert set(truck_efficiency["region"]) == VALID_ROAD_EDGE_REGIONS
assert set(transmission_efficiency["region"]) == VALID_PIPELINE_EDGE_REGIONS

print(f"Pipeline Efficiency rows: {len(pipeline_efficiency):,}")
print(f"Truck Efficiency rows: {len(truck_efficiency):,}")
print(f"Transmission Efficiency rows: {len(transmission_efficiency):,}")
print(f"Encoded Efficiency rows: {len(db_encoded['Efficiency']):,}")

In [ ]:
# =============================================================================
# Build transport CostVariable rows from transport_techs.csv
# =============================================================================

def build_transport_costvariable(
    links: pd.DataFrame,
    tech_specs: pd.DataFrame,
    notes: str,
) -> pd.DataFrame:

    rows = []

    assert links["distance_km"].notna().all()
    assert (links["distance_km"] > 0).all()
    assert links["canoe_region"].nunique() == len(links)

    required_cols = {
        "tech",
        "cost_per_km",
        "intercept_cost_per_km",
    }

    missing_cols = required_cols - set(tech_specs.columns)

    if missing_cols:
        raise ValueError(
            f"tech_specs is missing required columns: {sorted(missing_cols)}"
        )

    for tech in tech_specs.itertuples(index=False):

        cost_per_km = float(tech.cost_per_km)
        intercept_cost_per_km = float(tech.intercept_cost_per_km)

        df = pd.DataFrame(
            {
                "region": links["canoe_region"],
                "period": 1,
                "tech": tech.tech,
                "vintage": 1,
                "cost": (
                    intercept_cost_per_km
                    + cost_per_km * links["distance_km"]
                ),
                "units": "M$/unit",
                "notes": notes,
                "data_source": None,
                "dq_cred": None,
                "dq_geog": None,
                "dq_struc": None,
                "dq_tech": None,
                "dq_time": None,
                "data_id": "GEO001",
            }
        )

        rows.append(df)

    out = pd.concat(rows, ignore_index=True)

    assert (
        out[
            [
                "region",
                "period",
                "tech",
                "vintage",
                "data_id",
            ]
        ]
        .duplicated()
        .sum()
        == 0
    )

    return out


pipeline_costvariable = build_transport_costvariable(
    pipeline_links,
    pipeline_tech_specs,
    "Pipeline transport cost rebuilt from transport_techs.csv and selected graph-edge distance",
)

truck_costvariable = build_transport_costvariable(
    road_links,
    truck_tech_specs,
    f"Truck transport cost rebuilt from transport_techs.csv and selected {CONNECTION_METHOD} road-connected graph distance",
)

transmission_costvariable = build_transport_costvariable(
    pipeline_links,
    transmission_tech_specs,
    "Electricity transmission cost rebuilt from transport_techs.csv and selected graph-edge distance",
)

# Remove all inherited edge-region CostVariable rows.
# These are old baseline transport links and are not valid for the selected graph.
non_edge_costvariable = db_encoded["CostVariable"].loc[
    ~db_encoded["CostVariable"]["region"]
    .astype(str)
    .str.contains("-", regex=False)
].copy()

db_encoded["CostVariable"] = pd.concat(
    [
        non_edge_costvariable,
        pipeline_costvariable,
        truck_costvariable,
        transmission_costvariable,
    ],
    ignore_index=True,
)

assert set(pipeline_costvariable["region"]) == VALID_PIPELINE_EDGE_REGIONS
assert set(truck_costvariable["region"]) == VALID_ROAD_EDGE_REGIONS
assert set(transmission_costvariable["region"]) == VALID_PIPELINE_EDGE_REGIONS

print(f"Pipeline CostVariable rows: {len(pipeline_costvariable):,}")
print(f"Truck CostVariable rows: {len(truck_costvariable):,}")
print(f"Transmission CostVariable rows: {len(transmission_costvariable):,}")
print(f"Encoded CostVariable rows: {len(db_encoded['CostVariable']):,}")

display(
    db_encoded["CostVariable"]
    .loc[db_encoded["CostVariable"]["tech"].isin(TRANSPORT_TECHS)]
    .groupby("tech")["cost"]
    .describe()
)

In [ ]:
# =============================================================================
# Build and add zero truck CostInvest rows
# =============================================================================

truck_costinvest_rows = []

for truck in truck_tech_specs.itertuples(index=False):

    df = pd.DataFrame(
        {
            "region": road_links["canoe_region"],
            "tech": truck.tech,
            "vintage": 1,
            "cost": 0.0,
            "units": "M$/unit",
            "notes": "Existing road transport link; no road construction investment encoded",
            "data_source": None,
            "dq_cred": None,
            "dq_geog": None,
            "dq_struc": None,
            "dq_tech": None,
            "dq_time": None,
            "data_id": "GEO001",
        }
    )

    truck_costinvest_rows.append(df)

truck_costinvest = pd.concat(
    truck_costinvest_rows,
    ignore_index=True,
)

db_encoded["CostInvest"] = (
    db_encoded["CostInvest"]
    .loc[
        ~db_encoded["CostInvest"]["tech"].isin(TRUCK_TECHS)
    ]
    .copy()
)

db_encoded["CostInvest"] = pd.concat(
    [
        db_encoded["CostInvest"],
        truck_costinvest,
    ],
    ignore_index=True,
)

assert set(truck_costinvest["region"]) == VALID_ROAD_EDGE_REGIONS

print(f"Truck CostInvest rows: {len(truck_costinvest):,}")
print(f"Encoded CostInvest rows: {len(db_encoded['CostInvest']):,}")

In [ ]:
# =============================================================================
# Validate encoded transport-region coverage
# =============================================================================

for table_name in [
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:

    table = db_encoded[table_name].copy()

    if "region" not in table.columns:
        continue

    region_values = table["region"].dropna().astype(str)

    node_values = region_values[
        ~region_values.str.contains("-", regex=False)
    ]

    edge_values = region_values[
        region_values.str.contains("-", regex=False)
    ]

    invalid_node_regions = sorted(
        set(node_values) - VALID_NODE_REGIONS
    )

    valid_edge_regions = (
        VALID_PIPELINE_EDGE_REGIONS
        | VALID_ROAD_EDGE_REGIONS
    )

    invalid_edge_regions = sorted(
        set(edge_values) - valid_edge_regions
    )

    print(
        f"{table_name}: "
        f"{node_values.nunique():,} node regions, "
        f"{edge_values.nunique():,} edge regions, "
        f"{len(invalid_node_regions):,} invalid nodes, "
        f"{len(invalid_edge_regions):,} invalid edges"
    )

    assert not invalid_node_regions, (
        f"{table_name} has invalid node regions: "
        f"{invalid_node_regions[:10]}"
    )

    assert not invalid_edge_regions, (
        f"{table_name} has invalid edge regions: "
        f"{invalid_edge_regions[:10]}"
    )


pipeline_etl_regions = set(
    db_encoded["ETLSegment"]
    .loc[
        db_encoded["ETLSegment"]["tech_or_group"].isin(PIPE_TECHS),
        "region",
    ]
    .astype(str)
)

pipeline_eff_regions = set(
    db_encoded["Efficiency"]
    .loc[
        db_encoded["Efficiency"]["tech"].isin(PIPE_TECHS),
        "region",
    ]
    .astype(str)
)

assert pipeline_etl_regions == pipeline_eff_regions, (
    "Pipeline ETLSegment region coverage does not match "
    "pipeline Efficiency region coverage."
)

truck_etl_regions = set(
    db_encoded["ETLSegment"]
    .loc[
        db_encoded["ETLSegment"]["tech_or_group"].isin(TRUCK_TECHS),
        "region",
    ]
    .astype(str)
)

assert not truck_etl_regions, (
    "Truck technologies should have no ETLSegment rows, "
    f"but found regions: {sorted(truck_etl_regions)[:5]}"
)

print("Truck technologies confirmed to have no ETLSegment rows.")

print("\nPipeline ETLSegment coverage matches pipeline Efficiency coverage.")

In [ ]:
# =============================================================================
# Clear solver output tables
# =============================================================================

output_tables = [
    table_name
    for table_name in db_encoded
    if table_name.startswith("Output")
]

for table_name in output_tables:
    before_rows = len(db_encoded[table_name])
    db_encoded[table_name] = db_encoded[table_name].iloc[0:0].copy()
    print(f"{table_name}: {before_rows:,} → 0")

In [ ]:
# =============================================================================
# Final encoded database summary
# =============================================================================

final_summary = (
    pd.DataFrame(
        [
            {
                "table": table_name,
                "rows": len(df),
                "columns": len(df.columns),
            }
            for table_name, df in db_encoded.items()
        ]
    )
    .sort_values("table")
    .reset_index(drop=True)
)

display(final_summary)

In [ ]:
# =============================================================================
# Create fresh SQLite database from raw schema
# =============================================================================

if OUTPUT_SQLITE_PATH.exists():
    OUTPUT_SQLITE_PATH.unlink()

db_mgmt.convert_sql_to_sqlite(
    RAW_SCHEMA_PATH,
    OUTPUT_SQLITE_PATH,
)

print(f"Created fresh SQLite: {OUTPUT_SQLITE_PATH.name}")

In [ ]:
# =============================================================================
# Write encoded tables to fresh SQLite
# =============================================================================

db_mgmt.update_sqlite(
    OUTPUT_SQLITE_PATH,
    db_encoded,
)

print(f"Wrote {len(db_encoded)} tables to {OUTPUT_SQLITE_PATH.name}")

In [ ]:
# =============================================================================
# Verify exported SQLite database
# =============================================================================

db_test = db_mgmt.sqlite_to_dfs(
    OUTPUT_SQLITE_PATH,
)

print(f"Exported tables: {len(db_test)}")

for table_name in [
    "Region",
    "Technology",
    "Demand",
    "LimitCapacity",
    "Efficiency",
    "CostVariable",
    "CostInvest",
    "ETLSegment",
]:
    print(f"{table_name}: {len(db_test[table_name]):,}")

truck_techs = set(truck_tech_specs["tech"])
expected_truck_rows = len(road_links) * len(truck_tech_specs)

assert truck_techs.issubset(set(db_test["Technology"]["tech"]))
assert db_test["Efficiency"]["tech"].isin(truck_techs).sum() == expected_truck_rows
assert db_test["CostVariable"]["tech"].isin(truck_techs).sum() == expected_truck_rows
assert db_test["CostInvest"]["tech"].isin(truck_techs).sum() == expected_truck_rows

co2_cap_rows = db_test["LimitCapacity"].loc[
    db_test["LimitCapacity"]["tech_or_group"] == "CO2_CAP"
].copy()

co2_cap_positive = co2_cap_rows.loc[
    co2_cap_rows["capacity"] > 0
].copy()

print(f"CO2_CAP rows: {len(co2_cap_rows):,}")
print(f"CO2_CAP positive regions: {len(co2_cap_positive):,}")
print(f"Total CO2_CAP capacity: {co2_cap_rows['capacity'].sum():,.2f} t CO2e/year")

# co2_raw reflects only spatially assignable facilities (pre-filtered in NB8).
# Non-spatial exclusions are documented in co2_large_facilities_2024_metadata.csv.
print(f"Spatially assignable CO2 rows (from NB8 GPKG): {len(co2_raw):,}")
print(f"Dropped (zero/negative emissions): {len(co2_raw) - len(co2_facilities):,}")
print(f"Facilities entering snap: {len(co2_facilities):,}")

if "n_co2_facilities" in site_attributes.columns:
    print(f"Mapped CO2 facilities: {int(site_attributes['n_co2_facilities'].sum()):,}")
    print(f"Regions with mapped CO2 facilities: {(site_attributes['n_co2_facilities'] > 0).sum():,}")

assert len(co2_cap_rows) == len(db_test["Region"])
assert co2_cap_rows["capacity"].ge(0).all()
assert co2_cap_rows["capacity"].sum() > 0

print("\nExported SQLite validated.")